# Accelerometer vs Gyroscope (Paper Outputs Only)

This cleaned notebook keeps only the code required to generate:

- `UMAP_accer.pdf` (Figure 6)
- `dual_embedding_lift.mp4` (Figure 7)
- `accelr-clustering_results.tex` (Table 5)


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from sklearn.datasets import fetch_openml
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from scipy.linalg import eig
import umap

# Ensure imports work whether Jupyter runs from project root or notebooks/
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from functions import diffusion_map, LG_sym, calc_differential_vec

plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['mathtext.fontset'] = 'custom'
plt.rcParams['mathtext.rm'] = 'Times New Roman'
plt.rcParams['mathtext.it'] = 'Times New Roman:italic'
plt.rcParams['mathtext.bf'] = 'Times New Roman:bold'


In [ ]:
# Load Human Activity Recognition dataset
har = fetch_openml('har', version=1, as_frame=True)
X, y = har.data, har.target

# Keep original feature naming path used in the project
features_url = 'https://raw.githubusercontent.com/srvds/Human-Activity-Recognition/master/UCI_HAR_Dataset/features.txt'
features = pd.read_csv(features_url, sep=r'\s+', header=None, names=['idx', 'name'])
X.columns = features['name'].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_df = pd.DataFrame(X_scaled, columns=X.columns)

str_map = {
    '1': 'WALKING',
    '2': 'WALKING_UPSTAIRS',
    '3': 'WALKING_DOWNSTAIRS',
    '4': 'SITTING',
    '5': 'STANDING',
    '6': 'LAYING',
}

body_acc = X_df.filter(regex=r'tBodyAcc-')
gravity_acc = X_df.filter(regex=r'tGravityAcc-')

print('Body Acc shape:', body_acc.shape)
print('Gravity Acc shape:', gravity_acc.shape)


In [ ]:
# Core graph/eigendecomposition pipeline
P_ba, Q_ba, K_ba = diffusion_map(body_acc, adaptive=3000)
P_ga, Q_ga, K_ga = diffusion_map(gravity_acc, adaptive=3000)

L_ba, d_ba, v_ba = LG_sym(K_ba)
L_ga, d_ga, v_ga = LG_sym(K_ga)

s_body_unique, u_body_unique = calc_differential_vec(L_ba, v_ga, 3)
s_gravity_unique, u_gravity_unique = calc_differential_vec(L_ga, v_ba, 3)

L_shared = L_ba @ L_ga + L_ga @ L_ba
d_shared, v_shared = np.linalg.eigh(L_shared)
idx_shared = np.argsort(d_shared)[::-1]
v_shared = v_shared[:, idx_shared]


In [ ]:
# Shared plotting setup
reducer = umap.UMAP(
    n_components=2,
    n_neighbors=50,
    min_dist=0.001,
    random_state=30,
)

kmeans = KMeans(n_clusters=len(np.unique(y)), n_init=20, random_state=0)

y_str = [str_map[i] for i in y]

label_colors = {
    'WALKING': '#0B3C8C',            # deep blue
    'WALKING_UPSTAIRS': '#8E63CE',   # light purple
    'WALKING_DOWNSTAIRS': '#87CEFA', # light blue
    'SITTING': '#F2C300',            # yellow
    'STANDING': '#D1495B',           # red-pink
    'LAYING': '#2FA84F',             # green
}

activity_map = {
    'WALKING': 'Walk',
    'WALKING_UPSTAIRS': 'Walk-\nUpstairs',
    'WALKING_DOWNSTAIRS': 'Walk-\nDownstairs',
    'LAYING': 'Lay\n',
    'STANDING': 'Stand\n',
    'SITTING': 'Sit\n',
}


In [ ]:
# UMAP embeddings for requested figure
V_std_shared = StandardScaler().fit_transform(v_shared[:, :2])
V_umapshared = reducer.fit_transform(V_std_shared)

V_std_all = StandardScaler().fit_transform(np.c_[v_shared[:, :2], u_gravity_unique[:, :2], u_body_unique[:, :2]])
V_umapall = reducer.fit_transform(V_std_all)


In [ ]:
# Output 1: UMAP_accer.pdf
fig, ax = plt.subplots(1, 2, figsize=(9, 3.5))

for label, color in label_colors.items():
    idx = [i for i, v in enumerate(y_str) if v == label]

    ax[0].scatter(
        V_umapall[idx, 0],
        V_umapall[idx, 1],
        s=3,
        alpha=0.3,
        label=activity_map[label],
        color=color,
    )

    ax[1].scatter(
        V_umapshared[idx, 0],
        V_umapshared[idx, 1],
        s=3,
        alpha=0.4,
        color=color,
    )

ax[0].set_xlabel('UMAP 1', fontsize=16)
ax[0].set_ylabel('UMAP 2', fontsize=16)
ax[1].set_xlabel('UMAP 1', fontsize=16)
ax[1].set_ylabel('UMAP 2', fontsize=16)

handles, labels = ax[0].get_legend_handles_labels()
leg = fig.legend(
    handles,
    labels,
    bbox_to_anchor=(-0.03, 0.5),
    loc='center left',
    frameon=False,
    fontsize=14,
    markerscale=8,
)

for h in leg.legend_handles:
    h.set_alpha(0.9)

for a in ax:
    a.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

plt.tight_layout(rect=[0.15, 0, 1, 1])
plt.savefig('UMAP_accer.pdf')
plt.show()


In [ ]:
# Output 2: dual_embedding_lift.mp4
fig = plt.figure(figsize=(12, 6))
ax1 = fig.add_subplot(121, projection='3d')
ax2 = fig.add_subplot(122, projection='3d')

X1 = v_ba[:, 1]
Y1 = v_ba[:, 2]
Z1 = u_gravity_unique[:, 0]

X2 = v_ga[:, 1]
Y2 = v_ga[:, 2]
Z2 = u_body_unique[:, 0]

z_scale = 0.35
Z1 = z_scale * Z1 / np.max(np.abs(Z1))
Z2 = z_scale * Z2 / np.max(np.abs(Z2))

point_colors = [label_colors[lbl] for lbl in y_str]

scat1 = ax1.scatter(X1, Y1, np.zeros_like(Z1), c=point_colors, s=5, alpha=0.5)
scat2 = ax2.scatter(X2, Y2, np.zeros_like(Z2), c=point_colors, s=5, alpha=0.5)

ax1.set_title('Body + Gravity-specific', fontsize=26)
ax2.set_title('Gravity + Body-specific', fontsize=26)

for ax in [ax1, ax2]:
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_zticks([])
    ax.set_zlim(-z_scale, z_scale)
    ax.set_box_aspect([1, 1, 0.5])
    ax.view_init(elev=20, azim=50)

ax1.set_xlabel(r'$v^{Body}_1$', fontsize=23)
ax1.set_ylabel(r'$v^{Body}_2$', fontsize=23)
ax1.set_zlabel(r'$\delta^{Gravity}_0$', fontsize=23)

ax2.set_xlabel(r'$v^{Gravity}_1$', fontsize=23)
ax2.set_ylabel(r'$v^{Gravity}_2$', fontsize=23)
ax2.set_zlabel(r'$\delta^{Body}_0$', fontsize=23)

def update(frame):
    t = 0.5 * (1 - np.cos(np.pi * frame / 100))
    scat1._offsets3d = (X1, Y1, t * Z1)
    scat2._offsets3d = (X2, Y2, t * Z2)
    return scat1, scat2

anim = FuncAnimation(fig, update, frames=100, interval=40)
anim.save('dual_embedding_lift.mp4', dpi=300)
plt.close(fig)


## Clustering Comparison Table

In [ ]:
# DiLVE clustering metrics
ari_shared = adjusted_rand_score(y, kmeans.fit_predict(v_shared[:, :2]))
nmi_shared = normalized_mutual_info_score(y, kmeans.fit_predict(v_shared[:, :2]))

ari_b = adjusted_rand_score(y, kmeans.fit_predict(u_body_unique[:, :2]))
nmi_b = normalized_mutual_info_score(y, kmeans.fit_predict(u_body_unique[:, :2]))

ari_g = adjusted_rand_score(y, kmeans.fit_predict(u_gravity_unique[:, :2]))
nmi_g = normalized_mutual_info_score(y, kmeans.fit_predict(u_gravity_unique[:, :2]))

ari_all = adjusted_rand_score(y, kmeans.fit_predict(np.c_[v_shared[:, :2], u_gravity_unique[:, :2], u_body_unique[:, :2]]))
nmi_all = normalized_mutual_info_score(y, kmeans.fit_predict(np.c_[v_shared[:, :2], u_gravity_unique[:, :2], u_body_unique[:, :2]]))


In [ ]:
# Shnitzer et al. clustering metrics
S = P_ga @ Q_ba + P_ba @ Q_ga
D = P_ga @ Q_ba - P_ba @ Q_ga

ES_vals, VS = eig(S)
VS = VS[:, np.argsort(ES_vals)[::-1]]

EA_vals, VA = eig(D)
VA_imag = np.imag(VA[:, np.argsort(np.imag(EA_vals))[::-1]])
VA_real = np.real(VA[:, np.argsort(np.imag(EA_vals))[::-1]])

ari_shs = adjusted_rand_score(y, kmeans.fit_predict(VS[:, :2]))
nmi_shs = normalized_mutual_info_score(y, kmeans.fit_predict(VS[:, :2]))

ari_sh = adjusted_rand_score(y, kmeans.fit_predict(VA_imag[:, :2]))
nmi_sh = normalized_mutual_info_score(y, kmeans.fit_predict(VA_imag[:, :2]))

ari_shr = adjusted_rand_score(y, kmeans.fit_predict(VA_real[:, :2]))
nmi_shr = normalized_mutual_info_score(y, kmeans.fit_predict(VA_real[:, :2]))

ari_shall = adjusted_rand_score(y, kmeans.fit_predict(np.c_[VS[:, :2], VA_real[:, :2], VA_imag[:, :2]]))
nmi_shall = normalized_mutual_info_score(y, kmeans.fit_predict(np.c_[VS[:, :2], VA_real[:, :2], VA_imag[:, :2]]))


In [ ]:
# FKT clustering metrics
G_body = np.diag(np.sum(K_ba, axis=0)) - K_ba
G_gravity = np.diag(np.sum(K_ga, axis=0)) - K_ga

M_body = G_body + 1e-6 * np.eye(G_body.shape[0])
M_gravity = G_gravity + 1e-6 * np.eye(G_gravity.shape[0])

FK_body_unique = np.linalg.inv(M_body + M_gravity) @ M_gravity
FK_gravity_unique = np.linalg.inv(M_body + M_gravity) @ M_body

FK_values_1, body_unique_FK = eig(FK_body_unique)
FK_values_2, gravity_unique_FK = eig(FK_gravity_unique)

body_unique_FK = body_unique_FK[:, np.argsort(FK_values_1)[::-1]]
gravity_unique_FK = gravity_unique_FK[:, np.argsort(FK_values_2)[::-1]]

# Keep original workflow behavior for shared FKT clustering
s_fk, v_fk_shared = eig(FK_body_unique)
v_fk_shared = v_fk_shared[:, np.argsort(s_fk)[::-1]]

ari_fkts = adjusted_rand_score(y, kmeans.fit_predict(v_fk_shared[:, :2]))
nmi_fkts = normalized_mutual_info_score(y, kmeans.fit_predict(v_fk_shared[:, :2]))

ari_fktb = adjusted_rand_score(y, kmeans.fit_predict(body_unique_FK[:, :2]))
nmi_fktb = normalized_mutual_info_score(y, kmeans.fit_predict(body_unique_FK[:, :2]))

ari_fktg = adjusted_rand_score(y, kmeans.fit_predict(gravity_unique_FK[:, :2]))
nmi_fktg = normalized_mutual_info_score(y, kmeans.fit_predict(gravity_unique_FK[:, :2]))

ari_fktall = adjusted_rand_score(y, kmeans.fit_predict(np.c_[v_fk_shared[:, :2], body_unique_FK[:, :2], gravity_unique_FK[:, :2]]))
nmi_fktall = normalized_mutual_info_score(y, kmeans.fit_predict(np.c_[v_fk_shared[:, :2], body_unique_FK[:, :2], gravity_unique_FK[:, :2]]))


In [ ]:
# Output 3: accelr-clustering_results.tex
vectors = [r'$\psi^A$ \ real', r'$\psi^B$ \ imag', 'shared', 'all']
methods = ['DELVE', 'Shnitzer +', 'FKT']

ari = {
    r'$\psi^A$ \ real': {'DELVE': ari_b, 'Shnitzer +': ari_shr, 'FKT': ari_fktb},
    r'$\psi^B$ \ imag': {'DELVE': ari_g, 'Shnitzer +': ari_sh, 'FKT': ari_fktg},
    'shared': {'DELVE': ari_shared, 'Shnitzer +': ari_shs, 'FKT': ari_fkts},
    'all': {'DELVE': ari_all, 'Shnitzer +': ari_shall, 'FKT': ari_fktall},
}

nmi = {
    r'$\psi^A$ \ real': {'DELVE': nmi_b, 'Shnitzer +': nmi_shr, 'FKT': nmi_fktb},
    r'$\psi^B$ \ imag': {'DELVE': nmi_g, 'Shnitzer +': nmi_sh, 'FKT': nmi_fktg},
    'shared': {'DELVE': nmi_shared, 'Shnitzer +': nmi_shs, 'FKT': nmi_fkts},
    'all': {'DELVE': nmi_all, 'Shnitzer +': nmi_shall, 'FKT': nmi_fktall},
}

columns = pd.MultiIndex.from_product([methods, ['ARI', 'NMI']])
rows = []
for v in vectors:
    row = []
    for m in methods:
        row.extend([ari[v][m], nmi[v][m]])
    rows.append(row)

df = pd.DataFrame(rows, index=vectors, columns=columns)
df.index.name = 'Vector'

print(df)
df.to_latex('accelr-clustering_results.tex', escape=False, float_format='%.3f')
